# Experimento — SOM — organização bidimensional

**Aula 2 · 18/11 · Inteligência Artificial aplicada à Engenharia Elétrica**

## Antes de começar

### Objetivo
Observar 36 neurônios auto-organizáveis em um plano de corrente e velocidade normalizadas (adimensionais). Os dados são sintéticos.

### Pré-requisitos
Nenhum notebook anterior. Execute este arquivo de forma independente em uma sessão nova do Google Colab.

### Como executar
1. Abra o link público deste experimento no Colab: [Abrir SOM — organização bidimensional](https://colab.research.google.com/github/Furriel/code-library/blob/main/cursos/ia/redes-neurais/som-organizacao-2d.ipynb).
2. No menu **Ambiente de execução → Executar tudo**, rode as células de cima para baixo.
3. Faça a conta à mão antes de verificar a célula numérica.
4. Altere **somente o parâmetro sinalizado** no experimento guiado e execute tudo novamente.
5. Compare os gráficos e registre as conclusões; você não precisa enviar o notebook nem utilizar GitHub.

### Dados e parâmetros
A mudança da grade do estado inicial ao final e o erro de quantização sob duas taxas, sem comparar dados distintos.

O ambiente usa NumPy e Matplotlib disponíveis no Colab. O compilador C é opcional onde houver exemplo de referência.


## Cálculo manual — BMU e atualização

Considere `x=(0,2; 0,3)`, `m₁=(0,1; 0,1)` e `m₂=(0,8; 0,8)`. As distâncias quadráticas são `0,05` e `0,61`; a BMU é o neurônio 1. Com `α=0,2` e `h=1`, `m₁'=(0,12; 0,14)`. Faça a conta antes da conferência.


In [ ]:
import numpy as np
x=np.array([.2,.3]); m=np.array([[.1,.1],[.8,.8]])
d2=((m-x)**2).sum(axis=1)
bmu=int(np.argmin(d2)); novo=m[bmu]+.2*(x-m[bmu])
assert bmu==0 and np.allclose(d2,[.05,.61]) and np.allclose(novo,[.12,.14])
print(f"CÁLCULO MANUAL CONFERIDO: distâncias²={d2}; BMU={bmu+1}; peso novo={novo}")


In [ ]:
import json, math, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt

GCC = shutil.which("gcc") is not None
print("GCC disponível:", GCC)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

class LCG:
    # Mesmo gerador da página: s <- (1664525*s + 1013904223) mod 2^32; u = s / 2^32.
    def __init__(self, semente):
        self.s = semente % 2**32
    def proximo(self):
        self.s = (self.s * 1664525 + 1013904223) % 2**32
        return self.s / 4294967296
    def inteiro(self, n):
        return int(self.proximo() * n)

## 1. SOM em duas dimensões: ver a grade se desdobrar

Antes dos caracteres, um exemplo em que os pesos podem ser desenhados no próprio plano das entradas. Cada vértice da malha é o vetor de pesos de um neurônio; as linhas ligam vizinhos **na grade**. Os pontos vêm de dois regimes fictícios; as cores não entram no treino.

$$c=\arg\min_j\|x-m_j\|^2,\qquad h(c,j)=e^{-\|r_c-r_j\|^2/(2\sigma^2)},\qquad m_j\leftarrow m_j+\alpha\,h(c,j)\,(x-m_j)$$

In [ ]:
rng = np.random.default_rng(20261104)
dados2 = np.vstack([np.clip(rng.normal([.26, .31], .095, (60, 2)), .04, .96), np.clip(rng.normal([.73, .70], .095, (60, 2)), .04, .96)])
origem = np.array([0] * 60 + [1] * 60)
grade = 6
pos2 = np.array([(r, c) for r in range(grade) for c in range(grade)])
pesos2_0 = np.random.default_rng(8472).uniform(.12, .88, (grade * grade, 2)); pesos2 = pesos2_0.copy()
eq2 = []
for k in range(2000):
    x = dados2[k % len(dados2)]
    c = int(np.argmin(np.sum((pesos2 - x) ** 2, axis=1)))
    t = min(k / 1700, 1)
    alfa, sig = .28 * (.05 ** t), 2.5 * (.24 ** t)
    pesos2 += alfa * np.exp(-np.sum((pos2 - pos2[c]) ** 2, axis=1) / (2 * sig ** 2))[:, None] * (x - pesos2)
    if k % 50 == 0:
        eq2.append(np.mean(np.sqrt(np.min(((dados2[:, None] - pesos2[None]) ** 2).sum(2), axis=1))))
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for a, P, titulo in ((ax[0], pesos2_0, "Antes do treino"), (ax[1], pesos2, "Após 2000 amostras")):
    a.scatter(dados2[:, 0], dados2[:, 1], c=origem, s=12, alpha=.45, cmap="coolwarm")
    for j, (r, c) in enumerate(pos2):
        if c + 1 < grade: a.plot(*P[[j, j + 1]].T, "k-", lw=1, alpha=.6)
        if r + 1 < grade: a.plot(*P[[j, j + grade]].T, "k-", lw=1, alpha=.6)
    a.set(xlim=(0, 1), ylim=(0, 1), title=titulo, xlabel="corrente normalizada (adim.)", ylabel="velocidade normalizada (adim.)"); a.set_aspect("equal")
ax[2].plot(np.arange(0, 2000, 50), eq2, color="#146e83")
ax[2].set(title="Erro de quantização", xlabel="atualizações", ylabel="distância média à BMU (adim.)")
plt.tight_layout(); plt.show()

## Experimentos propostos — Experimento guiado: taxa inicial da SOM 2D

Altere somente `ALFA2_ALTERADA`. Os dois cenários usam exatamente os mesmos pontos, posições, pesos iniciais, ordem de apresentação e 2.000 atualizações. Registre as grades e curvas de erro no relatório único do Moodle.


In [ ]:
ALFA2_ORIGINAL = 0.28
ALFA2_ALTERADA = 0.14  # ALTERE SOMENTE ESTE VALOR (0 < alpha <= 1)
assert 0 < ALFA2_ALTERADA <= 1

def executar_som2d(alfa0):
    pesos=pesos2_0.copy()
    curva=[]
    for k in range(2000):
        x=dados2[k%len(dados2)]
        bmu=int(np.argmin(np.sum((pesos-x)**2,axis=1)))
        t=min(k/1700,1)
        a,sig=alfa0*(.05**t),2.5*(.24**t)
        pesos+=a*np.exp(-np.sum((pos2-pos2[bmu])**2,axis=1)/(2*sig**2))[:,None]*(x-pesos)
        if k%50==0:
            curva.append(float(np.mean(np.sqrt(np.min(((dados2[:,None]-pesos[None])**2).sum(2),axis=1)))))
    eq_final=float(np.mean(np.sqrt(np.min(((dados2[:,None]-pesos[None])**2).sum(2),axis=1))))
    return pesos,curva,eq_final

p0,curva0,eq0=executar_som2d(ALFA2_ORIGINAL)
p1,curva1,eq1=executar_som2d(ALFA2_ALTERADA)
assert np.allclose(p0,pesos2,atol=1e-12), "A reprodução original divergiu"
print("ANTES × DEPOIS — SOM 2D, mesmas 2.000 entradas")
print(f"Taxa {ALFA2_ORIGINAL:.2f}: EQ final={eq0:.6f}")
print(f"Taxa {ALFA2_ALTERADA:.2f}: EQ final={eq1:.6f}; diferença={eq1-eq0:+.6f}")
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(np.arange(0,2000,50),curva0,label=f"original alpha={ALFA2_ORIGINAL:.2f}")
ax.plot(np.arange(0,2000,50),curva1,label=f"alterada alpha={ALFA2_ALTERADA:.2f}")
ax.set(xlabel="atualizações",ylabel="EQ (adimensional)",title="Comparação controlada SOM 2D")
ax.legend();plt.tight_layout();plt.show()
print("INTERPRETAÇÃO AUTOMÁTICA:", "A taxa alterada reduziu a EQ final." if eq1 < eq0-1e-10 else "A EQ final não foi menor com a alteração.")
print("EQ não é medida de classificação nem de generalização. Registre a comparação no Moodle.")


## Exercícios para o relatório final

1. Desenhe no papel o exemplo da BMU e o deslocamento do vetor de pesos.
2. Compare duas taxas iniciais na mesma amostra e explique diferenças no erro de quantização e no formato final da grade.

Consulte os [exercícios e a rubrica](https://furriel.github.io/IA/apoio/exercicios.html).



## Questões de interpretação

A malha se ajustou aos dados? Uma EQ menor prova classificação correta? Registre o cálculo da BMU, as duas grades, as duas curvas EQ e a taxa no relatório único do Moodle.

## Conclusão

Este experimento é independente. Use o mesmo conjunto de dados e as mesmas condições iniciais ao comparar parâmetros. **Experimentos propostos:** altere apenas o valor indicado no código e justifique o resultado observado, sem afirmar melhora quando as métricas não a demonstram. Entregue somente um **único relatório final** pelo Moodle, conforme as instruções da disciplina.
